In [2]:
import pandas as pd

df = pd.read_csv("../../data/day_6.csv")

print(df.head())
print(df.shape)

      number            opened_at          resolved_at            closed_at  \
0  INC100000  2025-09-03 10:57:39  2025-09-07 16:02:56  2025-09-08 19:02:56   
1  INC100001  2025-09-26 10:56:16  2025-10-01 11:43:58  2025-10-02 08:43:58   
2  INC100002  2025-09-27 17:54:36  2025-10-01 22:33:03  2025-10-03 22:33:03   
3  INC100003  2025-09-20 23:28:09  2025-09-21 16:58:02  2025-09-24 02:58:02   
4  INC100004  2025-09-22 08:23:55  2025-09-24 22:23:37  2025-09-27 21:23:37   

              short_description  \
0              Batch job failed   
1     Suspicious email reported   
2  VPN disconnects during login   
3      Password reset requested   
4                       pls fix   

                                         description         category  \
0  Incident 100000: user reports vpn issue. Obser...       Facilities   
1  Incident 100001: user reports service portal i...               AV   
2  Incident 100002: user reports exchange issue. ...       HR Systems   
3  Incident 100003: us

In [3]:
print(df["made_sla"].value_counts())
print(df["made_sla"].value_counts(normalize=True) * 100)

made_sla
True     48000
False     2000
Name: count, dtype: int64
made_sla
True     96.0
False     4.0
Name: proportion, dtype: float64


In [4]:
df["sla_breach"] = (~df["made_sla"]).astype(int)

print(df["sla_breach"].value_counts())

sla_breach
0    48000
1     2000
Name: count, dtype: int64


In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = df["sla_breach"]
y_pred_baseline = [0] * len(df)

print("Accuracy :", accuracy_score(y_true, y_pred_baseline))
print("Precision:", precision_score(y_true, y_pred_baseline, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred_baseline, zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred_baseline, zero_division=0))

Accuracy : 0.96
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0


In [6]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_true, y_pred_baseline))

[[48000     0]
 [ 2000     0]]


In [7]:
df["opened_at"] = pd.to_datetime(df["opened_at"])

df["opened_hour"] = df["opened_at"].dt.hour
df["opened_day"] = df["opened_at"].dt.dayofweek
df["opened_month"] = df["opened_at"].dt.month

df["is_weekend"] = (df["opened_day"] >= 5).astype(int)
df["is_out_of_hours"] = ((df["opened_hour"] < 8) | (df["opened_hour"] >= 18)).astype(int)

print(df[["opened_at", "opened_hour", "opened_day", "is_out_of_hours"]].head())

            opened_at  opened_hour  opened_day  is_out_of_hours
0 2025-09-03 10:57:39           10           2                0
1 2025-09-26 10:56:16           10           4                0
2 2025-09-27 17:54:36           17           5                0
3 2025-09-20 23:28:09           23           5                1
4 2025-09-22 08:23:55            8           0                0


In [8]:
numeric_features = [
    "urgency",
    "impact",
    "opened_hour",
    "opened_day",
    "opened_month",
    "is_weekend",
    "is_out_of_hours"
]

categorical_features = [
    "category",
    "subcategory",
    "priority",
    "assignment_group",
    "contact_type",
    "requesting_department"
]

features = numeric_features + categorical_features

X = df[features]
y = df["sla_breach"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50000, 13)
y shape: (50000,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (40000, 13)
Testing : (10000, 13)


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [14]:
model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [15]:
y_pred = model.predict(X_test)

print(y_pred[:20])

[0 1 0 1 0 1 1 0 0 1 1 0 1 0 0 0 1 0 0 1]


In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.2%}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.2%}")
print(f"F1 Score : {f1_score(y_test, y_pred, zero_division=0):.2%}")

Accuracy : 71.33%
Precision: 8.48%
Recall   : 63.00%
F1 Score : 14.95%


In [17]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[6881 2719]
 [ 148  252]]


In [18]:
y_prob = model.predict_proba(X_test)[:, 1]

In [20]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_test, y_prob)

print(f"ROC-AUC: {roc_auc:.4f}")

ROC-AUC: 0.7098


In [21]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_prob)

print(f"PR-AUC: {pr_auc:.4f}")

PR-AUC: 0.1198


## Baseline vs Logistic Regression

In [23]:
y_true = df["sla_breach"]
y_pred_baseline = [0] * len(df)

baseline_accuracy = accuracy_score(y_true, y_pred_baseline)

In [24]:
print("Baseline Accuracy :", f"{baseline_accuracy:.2%}")
print("Model Accuracy    :", f"{accuracy_score(y_test, y_pred):.2%}")
print("Model Precision   :", f"{precision_score(y_test, y_pred):.2%}")
print("Model Recall      :", f"{recall_score(y_test, y_pred):.2%}")
print("Model F1 Score    :", f"{f1_score(y_test, y_pred):.2%}")
print("Model ROC-AUC     :", f"{roc_auc:.4f}")
print("Model PR-AUC      :", f"{pr_auc:.4f}")

Baseline Accuracy : 96.00%
Model Accuracy    : 71.33%
Model Precision   : 8.48%
Model Recall      : 63.00%
Model F1 Score    : 14.95%
Model ROC-AUC     : 0.7098
Model PR-AUC      : 0.1198


## Why 96% Accuracy Can Be Worthless

A model that predicts "no breach" for every incident achieves 96% accuracy because 96% of the tickets in this dataset do not breach SLA. However, this model completely fails to identify the 4% of tickets that actually breach SLA. Its recall for SLA breaches is 0%, meaning every actual breach is missed. This demonstrates why accuracy can be misleading when the target classes are highly imbalanced. For an SLA-breach prediction problem, the important objective is to identify tickets that are likely to breach so that the support team can intervene early. My Logistic Regression model has lower accuracy than the baseline, but it achieves 63% recall, meaning it catches 63% of the actual breaches in the test set. I would therefore not use accuracy as the main metric for this problem. I would focus on recall, precision, F1 and PR-AUC, with particular attention to recall because missing a real SLA breach can have greater operational consequences than investigating a ticket that turns out not to breach.

✅ Baseline: predict "No Breach"
✅ 96% baseline accuracy
✅ Logistic Regression
✅ Precision
✅ Recall
✅ F1
✅ Confusion Matrix
✅ ROC-AUC
✅ PR-AUC
✅ Baseline vs model comparison
✅ 150-word explanation
✅ False Positive vs False Negative probe